# Neural CFR+ traverser-action sampling

This notebook tests whether neural CFR+ can replace full traverser-action expansion with a fixed-size uniformly sampled subset.

The implementation:

- always evaluates `CALL` exactly when it is legal;
- samples claim actions uniformly without replacement;
- uses Horvitz–Thompson correction for sampled action values;
- inverse-weights records from deeper infosets by their traverser-action sampling reach;
- optionally uses the exact immediate-CALL payoff as a free control-variate baseline.

The pre-clipping regret estimator is unbiased. CFR+ clipping is nonlinear, so the complete update is not guaranteed to remain unbiased. Exact exploitability on this 18-claim game is therefore the deciding metric.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.eval.neural_evaluators import (
    AsyncExactExploitabilityEvaluator,
    ScheduledEvaluation,
)
from liars_poker.training.neural_runs import run_neural_cfr_plus

assert torch.cuda.is_available(), 'This experiment is intended for a CUDA machine.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## 1. Check the correction algebra

This synthetic test isolates the estimator from the game traversal. Both sampled estimators should recover the full action regrets before clipping. The CALL baseline should reduce variance when continuation values are correlated with the immediate-CALL value.

In [ ]:
rng = np.random.default_rng(7)
claim_count = 12
sample_count = 4
trials = 50_000

strategy = rng.dirichlet(np.ones(claim_count + 1))
call_value = -0.35
action_values = np.r_[call_value, call_value + rng.normal(0.0, 0.25, claim_count)]
exact_regret = action_values - strategy @ action_values
inclusion = sample_count / claim_count

def sampled_regret(baseline):
    selected = rng.choice(claim_count, size=sample_count, replace=False) + 1
    estimate = np.full(claim_count + 1, baseline, dtype=float)
    estimate[0] = call_value
    estimate[selected] = baseline + (action_values[selected] - baseline) / inclusion
    return estimate - strategy @ estimate

zero_samples = np.stack([sampled_regret(0.0) for _ in range(trials)])
call_samples = np.stack([sampled_regret(call_value) for _ in range(trials)])

estimator_check = pd.DataFrame({
    'max absolute bias': [
        np.max(np.abs(zero_samples.mean(axis=0) - exact_regret)),
        np.max(np.abs(call_samples.mean(axis=0) - exact_regret)),
    ],
    'mean action variance': [
        zero_samples.var(axis=0).mean(),
        call_samples.var(axis=0).mean(),
    ],
}, index=['zero baseline', 'CALL baseline'])
display(estimator_check.style.format(precision=6))

## 2. Equal-time online screen

`sample count` refers only to claim actions. CALL remains exact. The full-expansion control uses the current optimized CFR+ fitting schedule.

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

SEED = 7
MINUTES_PER_CONFIGURATION = 8
EVALUATE_EVERY_MINUTES = 2

CONFIGURATIONS = {
    'full expansion': {
        'sample_count': None,
        'baseline': 'none',
    },
    'sample 8; zero baseline': {
        'sample_count': 8,
        'baseline': 'none',
    },
    'sample 8; CALL baseline': {
        'sample_count': 8,
        'baseline': 'call',
    },
    'sample 4; zero baseline': {
        'sample_count': 4,
        'baseline': 'none',
    },
    'sample 4; CALL baseline': {
        'sample_count': 4,
        'baseline': 'call',
    },
}

BASE_TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}
TRAVERSALS_PER_PLAYER = 1024

screen_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
SCREEN_DIR = (
    REPO_ROOT / 'artifacts' / 'neural_method_screens'
    / f'cfr_plus_action_sampling_{spec.to_short_str()}___{screen_id}'
)
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
print('screen directory:', SCREEN_DIR)
print('planned measured GPU-training minutes:', len(CONFIGURATIONS) * MINUTES_PER_CONFIGURATION)

In [ ]:
def safe_name(name):
    return ''.join(ch if ch.isalnum() else '_' for ch in name).strip('_').lower()


def exact_schedule():
    return ScheduledEvaluation(
        name='exact_exploitability',
        evaluator=AsyncExactExploitabilityEvaluator(
            max_workers=1,
            compile_batch_size=65_536,
        ),
        every_minutes=EVALUATE_EVERY_MINUTES,
        run_at_end=True,
    )


def mean_validation(metrics, section, metric):
    values = [row[metric] for row in metrics[section] if row.get('records', 0)]
    return float(np.mean(values)) if values else np.nan


def normalized_auc(frame):
    ordered = frame.sort_values('measured_training_s')
    x = ordered['measured_training_s'].to_numpy(dtype=float)
    y = ordered['exploitability'].to_numpy(dtype=float)
    if len(x) < 2 or x[-1] <= x[0]:
        return np.nan
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))


completed_runs = {}


def run_configuration(name):
    if name in completed_runs:
        print('Already completed:', name)
        return completed_runs[name]['evaluations']
    config = CONFIGURATIONS[name]
    run_dir = SCREEN_DIR / safe_name(name)
    if run_dir.exists() and any(run_dir.iterdir()):
        raise FileExistsError(f'Run directory is not empty: {run_dir}')
    trainer_kwargs = {
        **BASE_TRAINER_KWARGS,
        'traverser_action_sample_count': config['sample_count'],
        'traverser_action_baseline': config['baseline'],
    }

    print(f'\n=== {name} ===')
    result = run_neural_cfr_plus(
        spec,
        minutes=MINUTES_PER_CONFIGURATION,
        traversals_per_player=TRAVERSALS_PER_PLAYER,
        trainer_kwargs=trainer_kwargs,
        evaluations=[exact_schedule()],
        run_dir=run_dir,
        save_checkpoint=False,
        wait_for_evaluations=True,
        debug=False,
    )

    training = pd.DataFrame(result.training_records)
    training['configuration'] = name
    evaluations = pd.DataFrame(result.evaluation_records)
    evaluations = evaluations[
        evaluations['evaluator'] == 'exact_exploitability'
    ].copy().sort_values('measured_training_s')
    evaluations['configuration'] = name

    validation = result.trainer.validation_metrics()
    diagnostics = {
        'final regret MSE': mean_validation(validation, 'regret', 'mse'),
        'final regret strategy TV': mean_validation(validation, 'regret', 'strategy_tv'),
        'final average-network TV': mean_validation(validation, 'strategy', 'strategy_tv'),
        'run directory': str(result.run_dir),
    }
    completed_runs[name] = {
        'training': training,
        'evaluations': evaluations,
        'diagnostics': diagnostics,
    }
    training.to_pickle(run_dir / 'screen_training.pkl')
    evaluations.to_csv(run_dir / 'screen_evaluations.csv', index=False)
    pd.DataFrame([diagnostics]).to_csv(run_dir / 'screen_diagnostics.csv', index=False)

    print(
        f'iterations={result.trainer.iteration} '
        f'train={result.measured_training_s / 60.0:.2f}m '
        f'evaluations={len(evaluations)}'
    )
    del result
    gc.collect()
    torch.cuda.empty_cache()
    return evaluations

In [ ]:
run_configuration('full expansion')

In [ ]:
run_configuration('sample 8; zero baseline')

In [ ]:
run_configuration('sample 8; CALL baseline')

In [ ]:
run_configuration('sample 4; zero baseline')

In [ ]:
run_configuration('sample 4; CALL baseline')

## 3. Results

The primary comparison is exact exploitability per measured neural-training minute. Edge fraction measures saved traversal work. Importance-weight ESS indicates how much variance the path correction introduces; values near one are healthy, while small values mean a few records dominate fitting.

In [ ]:
training_results = pd.concat(
    [run['training'] for run in completed_runs.values()],
    ignore_index=True,
)
evaluation_results = pd.concat(
    [run['evaluations'] for run in completed_runs.values()],
    ignore_index=True,
)

summary_rows = []
for name, run in completed_runs.items():
    training = run['training']
    evaluations = run['evaluations'].sort_values('measured_training_s')
    timing = pd.json_normalize(training['timing'])
    sampling = pd.json_normalize(training['action_sampling'])
    final = evaluations.iloc[-1]
    summary_rows.append({
        'configuration': name,
        'iterations completed': int(training['iteration'].iloc[-1]),
        'iterations / measured min': (
            training['iteration'].iloc[-1]
            / (training['measured_training_s'].iloc[-1] / 60.0)
        ),
        'final exploitability': final['exploitability'],
        'best exploitability': evaluations['exploitability'].min(),
        'normalized AUC': normalized_auc(evaluations),
        'mean traversal s': timing['traversal_s'].mean(),
        'mean regret fit s': timing['regret_training_s'].mean(),
        'mean strategy fit s': timing['strategy_training_s'].mean(),
        'mean claim-edge fraction': sampling['claim_edge_fraction'].mean(),
        'mean regret-weight ESS fraction': sampling['regret_weight_ess_fraction'].mean(),
        'largest regret weight': sampling['max_regret_weight'].max(),
        **run['diagnostics'],
    })

summary = pd.DataFrame(summary_rows).set_index('configuration')
summary = summary.sort_values('normalized AUC')
display(summary.drop(columns='run directory').style.format(precision=6))
summary.to_csv(SCREEN_DIR / 'summary.csv')
training_results.to_pickle(SCREEN_DIR / 'training_results.pkl')
evaluation_results.to_csv(SCREEN_DIR / 'evaluation_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

for name, frame in evaluation_results.groupby('configuration', sort=False):
    frame = frame.sort_values('measured_training_s')
    axes[0].plot(
        frame['measured_training_s'] / 60.0,
        frame['exploitability'],
        marker='o',
        label=name,
    )
    axes[1].plot(
        frame['iteration'],
        frame['exploitability'],
        marker='o',
        label=name,
    )

for ax, title, xlabel in [
    (axes[0], 'Exact exploitability by training time', 'Measured neural-training minutes'),
    (axes[1], 'Exact exploitability by CFR+ iteration', 'CFR+ iteration'),
]:
    ax.set(title=title, xlabel=xlabel, ylabel='Exploitability')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3)

axes[0].legend(fontsize=8)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

plot_summary = summary.drop(columns='run directory')
axes[0].bar(plot_summary.index, plot_summary['mean traversal s'])
axes[0].set(title='Traversal cost', ylabel='Seconds per iteration')

axes[1].bar(plot_summary.index, plot_summary['mean claim-edge fraction'])
axes[1].set(title='Traverser claim edges retained', ylabel='Fraction of full expansion')
axes[1].set_ylim(0, 1.05)

axes[2].bar(plot_summary.index, plot_summary['mean regret-weight ESS fraction'])
axes[2].set(title='Importance-weight effective sample size', ylabel='ESS fraction')
axes[2].set_ylim(0, 1.05)

for ax in axes:
    ax.tick_params(axis='x', labelrotation=30)
    ax.grid(axis='y', alpha=0.3)
fig.tight_layout()

## Interpretation rule

A useful sampled variant must improve exploitability per unit time, not merely iterations per minute. If the CALL baseline improves over the zero baseline at the same sample count, the control-variate direction is validated and a learned value baseline becomes the next experiment. If both sampled variants lose badly despite lower traversal cost, CFR+ clipping and/or large path weights are dominating the compute saving.